# Asistente de Abril Ideas Creativas
## Avance de proyecto 1

En este cuaderno preparé un chatbot basado en reglas para orientar consultas y reunir datos de una solicitud de impresión 3D.

**Cómo usarlo:** ejecuta las celdas en orden, con el botón de reproducción o con Mayús + Enter. Las explicaciones indican qué hace cada parte. Las salidas incluidas se generaron en Python local, no en Google Colab. Vuelve a ejecutarlo en Colab para obtener tu propia evidencia.

Los pedidos usados en las pruebas son ficticios. Los ejemplos de productos y la referencia a mensajes directos proceden de publicaciones indexadas del negocio. Precios, materiales, colores disponibles y plazos requieren confirmación de la propietaria.


### Paso 1 Preparar la biblioteca
NLTK permite asociar patrones con respuestas. Aquí lo uso para identificar la intención de una consulta. El programa no utiliza un modelo generativo ni aprende automáticamente de las conversaciones.

In [ ]:
%pip -q install nltk==3.9.2

In [1]:
import sys
import nltk
assert sys.version_info >= (3, 10), 'Se necesita Python 3.10 o superior.'
print('Python:', sys.version.split()[0])
print('NLTK:', nltk.__version__)

Python: 3.12.14
NLTK: 3.9.2


### Paso 2 Crear el programa
La celda guarda `asistente_3d.py`. `REGLAS` es la base de conocimiento: cada entrada tiene un nombre, un patrón y una respuesta. `normalizar` elimina diferencias entre mayúsculas y acentos para reconocer más variantes.

La clase `Asistente3D` mantiene el paso de una solicitud. Cuando escribo **cotizar**, pide producto, medidas, cantidad, color y fecha. Valida la cantidad y el formato de fecha; después registra los datos para revisión. El registro es local y no envía un pedido al negocio.

In [2]:
%%writefile asistente_3d.py
"""Avance 1: asistente de consultas y solicitudes de impresion 3D.
Python 3.10 o superior. Dependencia: nltk==3.9.2.
"""
import json
import re
import unicodedata
from datetime import date
from pathlib import Path
from nltk.chat.util import Chat


# Cada regla contiene una intencion, un patron y una respuesta.
REGLAS = [
    ("saludo", r".*\b(hola|buenos dias|buenas tardes|buenas noches)\b.*",
     "Hola. Soy el asistente de Abril Ideas Creativas. Puedo orientar tu solicitud 3D. Escribe ayuda para ver las opciones."),
    ("ayuda", r".*\b(ayuda|menu|opciones)\b.*",
     "Consulta productos, material, colores, medidas, diseno, archivos, precio, plazo, entrega o pago. Escribe cotizar para iniciar una solicitud."),
    ("productos", r".*\b(productos?|catalogo|que (hacen|venden|imprimen))\b.*",
     "El perfil publica ejemplos de llaveros tipo clicker, identificadores para lapices y figuras personalizadas. Consulta disponibilidad y viabilidad con la responsable."),
    ("material", r".*\b(material(es)?|pla|filamento)\b.*",
     "El material y su disponibilidad se confirman con la responsable al revisar el diseno y el uso de la pieza."),
    ("colores", r".*\b(colores?|tonos?)\b.*",
     "Indica el color que deseas. La disponibilidad se confirma al revisar tu solicitud."),
    ("medidas", r".*\b(medidas?|tamanos?|dimensiones?|centimetros?)\b.*",
     "Indica alto, ancho y largo en centimetros. El tamano final se revisa junto con el diseno."),
    ("archivos", r".*\b(archivos?|stl|3mf|foto|imagen)\b.*",
     "Si tienes un STL, 3MF o una imagen de referencia, mencionalo en la descripcion. La responsable confirmara si puede utilizarlo. Este prototipo no recibe adjuntos."),
    ("diseno", r".*\b(disenos?|personaliz\w*)\b.*",
     "El perfil anuncia pedidos personalizados. Describe tu idea para que la responsable revise si puede realizarse."),
    ("precio", r".*\b(precios?|costos?|cuanto cuesta|cuanto vale)\b.*",
     "El precio requiere revisar diseno, medidas y cantidad. Escribe cotizar para reunir esos datos; la cotizacion final la revisa el responsable."),
    ("plazo", r".*\b(plazos?|tiempos?|tardan?|demoran?|cuando estaria|para cuando)\b.*",
     "Indica para que fecha lo necesitas. El plazo se confirma despues de revisar el diseno, la cantidad y la disponibilidad."),
    ("entrega", r".*\b(entregas?|envios?|recoger|recogida|domicilio)\b.*",
     "La modalidad de entrega se acuerda con el responsable al revisar la cotizacion."),
    ("pago", r".*\b(pagos?|pagar|tarjetas?|efectivo|transferencia)\b.*",
     "La forma de pago se confirma con el responsable del negocio al revisar la cotizacion."),
    ("contacto", r".*\b(contacto|persona|responsable|hablar con alguien)\b.*",
     "El perfil invita a cotizar por mensaje directo. Consulta a Abril Ideas Creativas por sus redes. Este prototipo no envia mensajes al negocio."),
    ("resumen", r".*\b(resumen|ultima solicitud|ultimo pedido)\b.*", ""),
    ("cotizacion", r".*\b(cotiz\w*|presupuesto|nuevo pedido|solicitar pedido)\b.*", ""),
    ("cancelar", r"^(cancelar|cancela|cancelar solicitud)[.!? ]*$", ""),
    ("despedida", r"^(adios|gracias|salir|hasta luego)[.!? ]*$",
     "Gracias por consultar. Puedes volver a iniciar una solicitud cuando lo necesites."),
    ("desconocido", r".*",
     "No identifique tu consulta. Escribe ayuda para ver las opciones disponibles."),
]

CAMPOS = ["producto", "medidas", "cantidad", "color", "fecha"]
PREGUNTAS = [
    "Que producto necesitas?",
    "Que medidas necesitas? Indica alto, ancho y largo en cm.",
    "Cuantas piezas necesitas? Escribe un numero entero mayor que cero.",
    "Que color deseas?",
    "Para que fecha lo necesitas? Usa el formato AAAA-MM-DD.",
]


def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.strip().lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


class Asistente3D:
    def __init__(self, archivo=None):
        self.chat = Chat([(patron, [nombre]) for nombre, patron, _ in REGLAS])
        self.respuestas = {nombre: respuesta for nombre, _, respuesta in REGLAS}
        self.archivo = Path(archivo) if archivo else None
        self.solicitudes = []
        if self.archivo and self.archivo.exists():
            self.solicitudes = json.loads(self.archivo.read_text(encoding="utf-8"))
        self.pendiente = None
        self.ultima_intencion = ""

    def resumen(self, datos):
        return "; ".join(f"{campo}: {datos[campo]}" for campo in CAMPOS)

    def responder(self, texto):
        entrada = normalizar(texto)
        if not entrada:
            self.ultima_intencion = "entrada_vacia"
            return "Escribe una respuesta para continuar. Puedes escribir cancelar."
        intencion = self.chat.respond(entrada)
        self.ultima_intencion = intencion
        if intencion == "cancelar":
            self.pendiente = None
            return "Solicitud en curso cancelada. Puedes escribir cotizar para empezar otra."
        if self.pendiente is not None:
            paso = len(self.pendiente)
            if intencion in ("ayuda", "despedida"):
                return "Responde el dato solicitado o escribe cancelar. " + PREGUNTAS[paso]
            campo = CAMPOS[paso]
            valor = texto.strip()
            if campo == "cantidad":
                try:
                    valor = int(valor)
                    if valor <= 0:
                        raise ValueError
                except ValueError:
                    self.ultima_intencion = "error_cantidad"
                    return "Cantidad invalida. Escribe un numero entero mayor que cero."
            if campo == "fecha":
                try:
                    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
                        raise ValueError
                    valor = date.fromisoformat(valor).isoformat()
                except ValueError:
                    self.ultima_intencion = "error_fecha"
                    return "Fecha invalida. Usa una fecha real en formato AAAA-MM-DD."
            self.pendiente[campo] = valor
            self.ultima_intencion = "captura_" + campo
            if len(self.pendiente) < len(CAMPOS):
                return PREGUNTAS[len(self.pendiente)]
            self.solicitudes.append(self.pendiente.copy())
            if self.archivo:
                self.archivo.write_text(
                    json.dumps(self.solicitudes, ensure_ascii=False, indent=2),
                    encoding="utf-8")
            salida = "Solicitud registrada para revision. " + self.resumen(self.pendiente)
            self.pendiente = None
            return salida + ". Precio y fecha de entrega requieren confirmacion."
        if intencion == "cotizacion":
            self.pendiente = {}
            return PREGUNTAS[0]
        if intencion == "resumen":
            if self.solicitudes:
                return self.resumen(self.solicitudes[-1])
            return "Todavia no hay solicitudes registradas. Escribe cotizar."
        return self.respuestas[intencion]


if __name__ == "__main__":
    bot = Asistente3D("solicitudes_3d.json")
    print("Asistente 3D. Escribe ayuda o salir.")
    while True:
        mensaje = input("Tu: ")
        print("Bot:", bot.responder(mensaje))
        if normalizar(mensaje) == "salir" and bot.pendiente is None:
            break


Writing asistente_3d.py


### Paso 3 Probar una solicitud
Estos datos son un ejemplo de prueba. La fecha solicitada no significa que el negocio tenga disponibilidad. El archivo JSON conserva los registros mientras exista; en Colab hay que descargarlo para conservarlo fuera de la sesión.

In [3]:
from asistente_3d import Asistente3D
bot = Asistente3D('solicitudes_demo_3d.json')
ejemplo = ['Hola', 'Cotizar', 'Llaveros', '5 x 3 x 0.5 cm', '12', 'Azul', '2026-11-06', 'Resumen']
for mensaje in ejemplo:
    print('Usuario:', mensaje)
    print('Bot:', bot.responder(mensaje))
    print()

Usuario: Hola
Bot: Hola. Soy el asistente de Abril Ideas Creativas. Puedo orientar tu solicitud 3D. Escribe ayuda para ver las opciones.

Usuario: Cotizar
Bot: Que producto necesitas?

Usuario: Llaveros
Bot: Que medidas necesitas? Indica alto, ancho y largo en cm.

Usuario: 5 x 3 x 0.5 cm
Bot: Cuantas piezas necesitas? Escribe un numero entero mayor que cero.

Usuario: 12
Bot: Que color deseas?

Usuario: Azul
Bot: Para que fecha lo necesitas? Usa el formato AAAA-MM-DD.

Usuario: 2026-11-06
Bot: Solicitud registrada para revision. producto: Llaveros; medidas: 5 x 3 x 0.5 cm; cantidad: 12; color: Azul; fecha: 2026-11-06. Precio y fecha de entrega requieren confirmacion.

Usuario: Resumen
Bot: producto: Llaveros; medidas: 5 x 3 x 0.5 cm; cantidad: 12; color: Azul; fecha: 2026-11-06



### Paso 4 Ejecutar diez conversaciones distintas
Las conversaciones comprueban consultas de productos, material, color, medidas, archivos, personalización, precio, plazo, entrega, pago, contacto, captura de datos, cancelación y recuperación de errores.

Las verificaciones también comprueban que una solicitud cancelada no se registra, que una cantidad inválida no avanza al siguiente campo y que los datos guardados se recuperan al abrir otra instancia. Después se exploran tres limitaciones.

In [4]:
%%writefile pruebas_3d.py
"""Diez conversaciones funcionales y tres exploraciones de limites.
Ejecutar con Python 3.10 o superior: python pruebas_3d.py
"""
import json
import platform
import tempfile
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import nltk
from asistente_3d import Asistente3D

try:
    import google.colab
    ENTORNO = "Google Colab"
except ImportError:
    ENTORNO = "Python local; no corresponde a una ejecucion en Google Colab"


CONVERSACIONES = [
    ("C01", "Saludo y productos", [
        ("¡Hola!", "saludo"),
        ("¿Qué productos imprimen?", "productos")]),
    ("C02", "Material y color", [
        ("¿Qué material usan?", "material"),
        ("¿Qué colores tienen?", "colores")]),
    ("C03", "Medidas y archivos", [
        ("¿Qué medidas debo indicar?", "medidas"),
        ("¿Aceptan archivos STL?", "archivos")]),
    ("C04", "Personalización y precio", [
        ("¿Puedo mandar mi propio diseño?", "diseno"),
        ("¿Cuánto cuesta?", "precio")]),
    ("C05", "Plazo y entrega", [
        ("¿Cuánto tardan?", "plazo"),
        ("¿Hacen envíos?", "entrega")]),
    ("C06", "Pago y revisión humana", [
        ("¿Cómo puedo pagar?", "pago"),
        ("Quiero hablar con alguien", "contacto")]),
    ("C07", "Ayuda y solicitud completa", [
        ("Ayuda", "ayuda"),
        ("Quiero cotizar", "cotizacion"),
        ("Llaveros", "captura_producto"),
        ("5 x 3 x 0.5 cm", "captura_medidas"),
        ("12", "captura_cantidad"),
        ("Azul", "captura_color"),
        ("2026-11-06", "captura_fecha"),
        ("Muéstrame el resumen", "resumen")]),
    ("C08", "Cancelación y reinicio", [
        ("Nuevo pedido", "cotizacion"),
        ("Figuras", "captura_producto"),
        ("Cancelar", "cancelar"),
        ("Resumen", "resumen")]),
    ("C09", "Validación de cantidad y fecha", [
        ("Cotizar", "cotizacion"),
        ("Recuerdos", "captura_producto"),
        ("8 x 5 x 2 cm", "captura_medidas"),
        ("0", "error_cantidad"),
        ("5", "captura_cantidad"),
        ("Blanco", "captura_color"),
        ("2026-02-30", "error_fecha"),
        ("2026-11-10", "captura_fecha")]),
    ("C10", "Entrada vacía y consulta no cubierta", [
        ("", "entrada_vacia"),
        ("¿Quién ganó el partido?", "desconocido"),
        ("Gracias", "despedida")]),
]


def ejecutar_pruebas(directorio="evidencias"):
    destino = Path(directorio)
    destino.mkdir(parents=True, exist_ok=True)
    resultados = []
    lineas = ["PRUEBAS REALES: " + ENTORNO, "Datos de pedidos ficticios de prueba."]
    for codigo, titulo, turnos in CONVERSACIONES:
        bot = Asistente3D()
        registro = {"id": codigo, "titulo": titulo, "turnos": []}
        lineas.append(f"\n{codigo} | {titulo}")
        for entrada, esperado in turnos:
            salida = bot.responder(entrada)
            correcto = bot.ultima_intencion == esperado
            registro["turnos"].append({
                "entrada": entrada, "intencion_esperada": esperado,
                "intencion_observada": bot.ultima_intencion,
                "respuesta": salida, "correcto": correcto})
            lineas.extend([f"Usuario: {entrada or '[entrada vacia]'}", f"Bot: {salida}"])
            assert correcto, f"{codigo}: {entrada!r} produjo {bot.ultima_intencion}"
        if codigo == "C07":
            assert bot.solicitudes[0]["cantidad"] == 12
            assert len(bot.solicitudes) == 1
            assert "Llaveros" in registro["turnos"][-1]["respuesta"]
        if codigo == "C08":
            assert not bot.solicitudes and bot.pendiente is None
        if codigo == "C09":
            assert len(bot.solicitudes) == 1
            assert bot.solicitudes[0]["cantidad"] == 5
            assert bot.solicitudes[0]["fecha"] == "2026-11-10"
        registro["correcta"] = all(t["correcto"] for t in registro["turnos"])
        resultados.append(registro)

    # Verificar que una solicitud completa se conserva al abrir otra sesion.
    with tempfile.TemporaryDirectory() as carpeta:
        archivo = Path(carpeta) / "solicitudes.json"
        bot = Asistente3D(archivo)
        for entrada in ["cotizar", "Llavero", "5 x 3 x 1 cm", "2", "Rosa", "2026-11-06"]:
            bot.responder(entrada)
        assert Asistente3D(archivo).solicitudes[0]["color"] == "Rosa"

    limites = []
    for entrada, observacion in [
        ("¿Qué colores tienen y cuánto cuesta?", "Responde sobre colores y omite el precio: una sola intencion por turno."),
        ("kiero unos yaveros", "No reconoce errores de escritura que no estan en las reglas."),
        ("Lo mismo que la vez pasada", "No interpreta referencias a conversaciones anteriores.")]:
        bot = Asistente3D()
        salida = bot.responder(entrada)
        limites.append({"entrada": entrada, "respuesta": salida,
                        "intencion": bot.ultima_intencion, "observacion": observacion})
        lineas.extend(["\nLIMITE OBSERVADO", f"Usuario: {entrada}", f"Bot: {salida}", observacion])
    resumen = {
        "fecha_ejecucion": datetime.now(ZoneInfo("America/Chicago")).isoformat(),
        "entorno": ENTORNO,
        "python": platform.python_version(), "nltk": nltk.__version__,
        "conversaciones": len(resultados),
        "conversaciones_correctas": sum(r["correcta"] for r in resultados),
        "turnos": sum(len(r["turnos"]) for r in resultados),
        "persistencia_json_verificada": True,
        "resultados": resultados, "limites": limites,
    }
    (destino / "resultados.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
    (destino / "conversaciones.txt").write_text("\n".join(lineas), encoding="utf-8")
    print(f"{resumen['conversaciones_correctas']}/{len(resultados)} conversaciones correctas; {resumen['turnos']} turnos.")
    print("Persistencia JSON verificada. Tres limites explorados. Evidencia:", destino.resolve())
    return resumen


if __name__ == "__main__":
    ejecutar_pruebas()


Writing pruebas_3d.py


In [5]:
from pathlib import Path
from pruebas_3d import ejecutar_pruebas
resultado = ejecutar_pruebas()
print('Entorno observado:', resultado['entorno'])
print(Path('evidencias/conversaciones.txt').read_text(encoding='utf-8'))

10/10 conversaciones correctas; 35 turnos.
Persistencia JSON verificada. Tres limites explorados. Evidencia: /workspace/scratch/21319f4e9f45/qa/notebook_inprocess/evidencias
Entorno observado: Python local; no corresponde a una ejecucion en Google Colab
PRUEBAS REALES: Python local; no corresponde a una ejecucion en Google Colab
Datos de pedidos ficticios de prueba.

C01 | Saludo y productos
Usuario: ¡Hola!
Bot: Hola. Soy el asistente de Abril Ideas Creativas. Puedo orientar tu solicitud 3D. Escribe ayuda para ver las opciones.
Usuario: ¿Qué productos imprimen?
Bot: El perfil publica ejemplos de llaveros tipo clicker, identificadores para lapices y figuras personalizadas. Consulta disponibilidad y viabilidad con la responsable.

C02 | Material y color
Usuario: ¿Qué material usan?
Bot: El material y su disponibilidad se confirman con la responsable al revisar el diseno y el uso de la pieza.
Usuario: ¿Qué colores tienen?
Bot: Indica el color que deseas. La disponibilidad se confirma al r

### Paso 5 Conversar manualmente
Esta función permite escribir preguntas en la consola del cuaderno. Para iniciarla, quita el símbolo `#` de la última línea y ejecuta la celda. Escribe **cancelar** para abandonar una solicitud en curso y **salir** para terminar la conversación.

In [6]:
from asistente_3d import normalizar
def conversar():
    asistente = Asistente3D('solicitudes_3d.json')
    print('Asistente de Abril Ideas Creativas. Escribe ayuda o salir.')
    while True:
        entrada = input('Tú: ')
        print('Bot:', asistente.responder(entrada))
        if normalizar(entrada) == 'salir' and asistente.pendiente is None:
            break

# conversar()

### Paso 6 Conservar las evidencias
Después de ejecutar las pruebas en Colab, guarda capturas donde se vean el nombre del cuaderno, entradas y respuestas. Descarga `evidencias/conversaciones.txt` y `evidencias/resultados.json` desde el panel de archivos. Guarda el cuaderno con sus salidas.

La función siguiente es opcional: en Colab puedes llamarla para descargar un archivo. Si registras solicitudes fuera de las pruebas, descarga también su JSON. La sesión de Colab es temporal.

In [7]:
def descargar_evidencia(nombre='evidencias/resultados.json'):
    from google.colab import files
    files.download(nombre)

# descargar_evidencia()
# descargar_evidencia('evidencias/conversaciones.txt')

### Paso 7 Guardar el código en GitHub
Conserva en tu repositorio este cuaderno, `asistente_3d.py`, `pruebas_3d.py`, `requirements.txt` y las evidencias de prueba. En el documento del avance añade los enlaces reales de Colab y GitHub y comprueba que el docente tenga acceso.

### Qué puedo explicar del código
- Una intención es el motivo de la pregunta, como consultar precio o iniciar una cotización.
- Un patrón es una regla que busca palabras o expresiones en la entrada.
- La base de conocimiento contiene las reglas y sus respuestas.
- El estado indica qué dato falta para completar una solicitud.
- El JSON es un archivo que guarda los datos; no es una base de datos conectada a las redes sociales.

### Referencias
- NLTK Project. `nltk.chat.util`: https://www.nltk.org/api/nltk.chat.util.html
- Google. Preguntas frecuentes de Colab: https://research.google.com/colaboratory/faq.html
- GitHub Docs. Adding a file to a repository: https://docs.github.com/en/repositories/working-with-files/managing-files/adding-a-file-to-a-repository
- Abril Ideas Creativas: https://www.facebook.com/abrilideascreativas/ y https://www.instagram.com/abril.ideascreativas/
